In [ ]:
import networkx as nx
import pickle
import numpy as np
from matplotlib import pyplot as plt
from shapely import affinity, Polygon, Point
import folium
import geopandas as gpd

from math import radians, cos, sin, asin, sqrt


In [ ]:
with open('../data/fire_graph_4.pkl', 'rb') as handle:
    G = pickle.load(handle)

In [ ]:
with open('../data/results/forest_separation_varying_tolerance_2024-02-12_2.pkl', 'rb') as f:
    varying_tol_dict = pickle.load(f)

In [ ]:
with open('../data/results/forest_separation_ridge_line_2024-02-12.pkl', 'rb') as f:
    ridge_line_dict = pickle.load(f)

In [ ]:
keys = list(varying_tol_dict.keys())
keys.sort()
keys

In [ ]:
Gcc = sorted(nx.connected_components(G), key=len, reverse=True)
largest_connected_component = Gcc[0]
G0 = G.subgraph(largest_connected_component)

total = len(largest_connected_component)

In [ ]:
updated_results = {}

In [ ]:
tolerance = list(keys)[36]
print(tolerance)

group_1 = varying_tol_dict[tolerance]['group_1']
group_2 = varying_tol_dict[tolerance]['group_2']
sep_group = varying_tol_dict[tolerance]['sep_group']

for node in sep_group:
    connected_to = []

    for edges in G.edges(node):
        if edges[0] == node:
            if edges[1] in group_1:
                connected_to.append('group_1')
            elif edges[1] in group_2:
                connected_to.append('group_2')
        if edges[1] == node:
            if edges[0] in group_1:
                connected_to.append('group_1')
            elif edges[0] in group_2:
                connected_to.append('group_2')

    if len(connected_to) > 0:
        if not ('group_1' in connected_to and 'group_2' in connected_to):
            print(node, connected_to)
            if connected_to[0] == 'group_1':
                group_1.append(node)
                sep_group.remove(node)
                print('added to group 1') 

            if connected_to[0] == 'group_2':
                group_2.append(node)
                sep_group.remove(node)
                print('added to group 2')  


In [ ]:
illegal_edges = []

for u, v in G.edges:
    if u in group_1:
        xu0 = 1
        xu1 = 0
    elif u in group_2:
        xu0 = 0
        xu1 = 1 
    else:
        xu0 = 0
        xu1 = 0 
        
    if v in group_1:
        xv0 = 1
        xv1 = 0
    elif v in group_2:
        xv0 = 0
        xv1 = 1 
    else:
        xv0 = 0
        xv1 = 0 
        
    if xu0*xv1 == 1 or xu1*xv0 == 1:
        illegal_edges.append((u,v))
        print(u,v)
        print(xu0,xv1, xv0,xu1)
            
print("\nNumber of illegal edges:\t", len(illegal_edges))


In [ ]:
updated_results.update({tolerance:{'group_1':group_1, 'group_2':group_2, 'sep_group':sep_group}})

In [ ]:
len(group_1), len(group_2), len(sep_group)

In [ ]:
len(updated_results[0]['sep_group'])

In [ ]:
len_group_1 = []
len_group_2 = []
len_sep_group = []
group_diff = []
len_sep_group_all = []
group_diff_all = []

tolerances = []

for result in list(updated_results.keys()):
    separator_group = list(set(updated_results[result]['sep_group']) & set(largest_connected_component))
    group1 = list(set(updated_results[result]['group_1']) & set(largest_connected_component))
    group2 = list(set(updated_results[result]['group_2']) & set(largest_connected_component))
    
    len_sep_group_all.append(len(separator_group))
    group_diff_all.append(np.abs(len(group1) - len(group2)))

    if len(len_sep_group) == 0:
        len_sep_group.append(len(separator_group))
        group_diff.append(np.abs(len(group1) - len(group2)))
        len_group_1.append(len(group1))
        len_group_2.append(len(group2))
        tolerances.append(result)
    else:
        if ((len_sep_group[-1] - len(separator_group)) > 0):
            len_sep_group.append(len(separator_group))
            group_diff.append(np.abs(len(group1) - len(group2)))
            len_group_1.append(len(group1))
            len_group_2.append(len(group2))
            tolerances.append(result)

## Separator Size vs Group Difference

In [ ]:
plt.figure(figsize=(5, 5), dpi=80)
plt.rc('grid', linestyle="--", color='lightgrey')
plt.grid(True)


plt.scatter(group_diff_all,len_sep_group_all, color = 'teal', edgecolors='darkgrey', s = 70, alpha = 0.25, label='All Results')
plt.scatter(group_diff,len_sep_group, color = 'teal', edgecolors='darkgrey', s = 70, label='Minima')
plt.scatter([group_diff[2]],[len_sep_group[2]], color = 'indianred', edgecolors='darkgrey', s = 70, label='11 separator example')
plt.scatter([group_diff[4]],[len_sep_group[4]], color = 'darksalmon', edgecolors='darkgrey', s = 70, label='6 separator example')

plt.xlabel('Difference in Group Size')
plt.ylabel('Number of Separators')
#plt.title('Separators Size with Varying Tolerance')
plt.legend()

plt.savefig('../figures/separator_size_vs_group_difference.png', bbox_inches='tight')


## Detailed Results for Each

In [ ]:
def get_poly_sqr_from_bounds(active_zone_bounds):
    
    active_zone_ll = Polygon([[active_zone_bounds[0][0],active_zone_bounds[1][0]],
                              [active_zone_bounds[0][1],active_zone_bounds[1][0]],
                              [active_zone_bounds[0][1],active_zone_bounds[1][1]],
                              [active_zone_bounds[0][0],active_zone_bounds[1][1]],
                              [active_zone_bounds[0][0],active_zone_bounds[1][0]]])

    return active_zone_ll


### 11 Separators

In [ ]:
len_sep_group_all[6]

In [ ]:
active_zone_bounds =  ((-122.00, -121.75),(37.00, 37.25))
active_zone_ll = get_poly_sqr_from_bounds(active_zone_bounds)
lon, lat = active_zone_ll.centroid.coords[0]
zoom_start_val = 11

# Basemap
m = folium.Map([lat, lon], tiles='cartodbpositron', zoom_start=zoom_start_val)#, tiles='OpenStreetMap')

# Bounding box around active zone = Red
sim_geo = gpd.GeoSeries(active_zone_ll).simplify(tolerance=0.001)
geo_j = sim_geo.to_json()
geo_j = folium.GeoJson(data=geo_j, style_function=lambda x: {"fill":False,"color": "red"})#orange"})
geo_j.add_to(m)

for edge in G0.edges():
    c1 = G.nodes[edge[0]]['pos']
    c2 = G.nodes[edge[1]]['pos']
    folium.PolyLine([[c1[1],c1[0]],[c2[1],c2[0]]], weight=1).add_to(m)

for node in G0.nodes():
    if node in updated_results[tolerances[2]]['sep_group']:
        folium.CircleMarker(location=[G.nodes[node]['pos'][1], G.nodes[node]['pos'][0]],
                            radius=4,
                            color = 'black', tooltip= str(node)+ ' '+ str([G.nodes[node]['pos'][1], G.nodes[node]['pos'][0]])).add_to(m)  #weight=5, color = 'green').add_to(m)
    
        folium.CircleMarker(location=[G.nodes[node]['pos'][1], G.nodes[node]['pos'][0]],
                            radius=0.2,
                            color = 'darkorange', tooltip= str(node)+ ' '+ str([G.nodes[node]['pos'][1], G.nodes[node]['pos'][0]])).add_to(m)  #weight=5, color = 'green').add_to(m)
    
    elif node in updated_results[tolerances[2]]['group_1']:
        folium.CircleMarker(location=[G.nodes[node]['pos'][1], G.nodes[node]['pos'][0]],
                            radius=0.1,
                            color = 'blue', tooltip= str(node) + ' '+ str([G.nodes[node]['pos'][1], G.nodes[node]['pos'][0]])).add_to(m)  #weight=5, color = 'green').add_to(m)

    elif node in updated_results[tolerances[2]]['group_2']:
        folium.CircleMarker(location=[G.nodes[node]['pos'][1], G.nodes[node]['pos'][0]],
                            radius=0.1,
                            color = 'aqua', tooltip= str(node) + ' '+ str([G.nodes[node]['pos'][1], G.nodes[node]['pos'][0]])).add_to(m)  #weight=5, color = 'green').add_to(m)

print('11 Separators')
m

In [ ]:
print('11 separators\n')

print('group 1 size: {:d}, \ngroup 2 size: {:d}, \nseparator group size: {:d}\n'.format(len_group_1[2],len_group_2[2],len_sep_group[2]))


print('group 1 %: {:0.2f}, \ngroup 2 %: {:0.2f}, \nseparator group %: {:0.2f}\n'.format(100*(len_group_1[2]/total),
                                                                                        100*(len_group_2[2]/total),
                                                                                        100*(len_sep_group[2]/total)))

print('% diff in group size: {:0.2f}, \nnumber difference: {:d}, \nN spearators: {:d}'.format(100*(len_group_1[2] - len_group_2[2])/total, 
                                                                                              len_group_1[2] - len_group_2[2], 
                                                                                              len_sep_group[2]))


### 6 Separators

In [ ]:
active_zone_bounds =  ((-122.00, -121.75),(37.00, 37.25))
active_zone_ll = get_poly_sqr_from_bounds(active_zone_bounds)
lon, lat = active_zone_ll.centroid.coords[0]
zoom_start_val = 11

# Basemap
m = folium.Map([lat, lon], tiles='cartodbpositron', zoom_start=zoom_start_val)#, tiles='OpenStreetMap')

# Bounding box around active zone = Red
sim_geo = gpd.GeoSeries(active_zone_ll).simplify(tolerance=0.001)
geo_j = sim_geo.to_json()
geo_j = folium.GeoJson(data=geo_j, style_function=lambda x: {"fill":False,"color": "red"})#orange"})
geo_j.add_to(m)

for edge in G0.edges():
    c1 = G.nodes[edge[0]]['pos']
    c2 = G.nodes[edge[1]]['pos']
    folium.PolyLine([[c1[1],c1[0]],[c2[1],c2[0]]], weight=1).add_to(m)

for node in G0.nodes():
    if node in updated_results[tolerances[4]]['sep_group']:
        folium.CircleMarker(location=[G.nodes[node]['pos'][1], G.nodes[node]['pos'][0]],
                            radius=4,
                            color = 'black', tooltip= str(node)+ ' '+ str([G.nodes[node]['pos'][1], G.nodes[node]['pos'][0]])).add_to(m)  #weight=5, color = 'green').add_to(m)

    
        folium.CircleMarker(location=[G.nodes[node]['pos'][1], G.nodes[node]['pos'][0]],
                            radius=0.1,
                            color = 'darkorange', tooltip= str(node)+ ' '+ str([G.nodes[node]['pos'][1], G.nodes[node]['pos'][0]])).add_to(m)  #weight=5, color = 'green').add_to(m)
    
    elif node in updated_results[tolerances[4]]['group_1']:
        folium.CircleMarker(location=[G.nodes[node]['pos'][1], G.nodes[node]['pos'][0]],
                            radius=0.1,
                            color = 'blue', tooltip= str(node) + ' '+ str([G.nodes[node]['pos'][1], G.nodes[node]['pos'][0]])).add_to(m)  #weight=5, color = 'green').add_to(m)

    elif node in updated_results[tolerances[4]]['group_2']:
        folium.CircleMarker(location=[G.nodes[node]['pos'][1], G.nodes[node]['pos'][0]],
                            radius=0.1,
                            color = 'aqua', tooltip= str(node) + ' '+ str([G.nodes[node]['pos'][1], G.nodes[node]['pos'][0]])).add_to(m)  #weight=5, color = 'green').add_to(m)


print('6 separators')
m

In [ ]:
print('6 separators\n')

print('group 1 size: {:d}, \ngroup 2 size: {:d}, \nseparator group size: {:d}\n'.format(len_group_1[4],len_group_2[4],len_sep_group[4]))


print('group 1 %: {:0.2f}, \ngroup 2 %: {:0.2f}, \nseparator group %: {:0.2f}\n'.format(100*(len_group_1[4]/total),
                                                                                        100*(len_group_2[4]/total),
                                                                                        100*(len_sep_group[4]/total)))

print('% diff in group size: {:0.2f}, \nnumber difference: {:d}, \nN spearators: {:d}'.format(100*(len_group_1[4] - len_group_2[4])/total, 
                                                                                              len_group_1[4] - len_group_2[4], 
                                                                                              len_sep_group[4]))


### Ridge Line Separation

In [ ]:
active_zone_bounds =  ((-122.00, -121.75),(37.00, 37.25))
active_zone_ll = get_poly_sqr_from_bounds(active_zone_bounds)
lon, lat = active_zone_ll.centroid.coords[0]
zoom_start_val = 11

# Basemap
m = folium.Map([lat, lon], tiles='cartodbpositron', zoom_start=zoom_start_val)#, tiles='OpenStreetMap')

# Bounding box around active zone = Red
sim_geo = gpd.GeoSeries(active_zone_ll).simplify(tolerance=0.001)
geo_j = sim_geo.to_json()
geo_j = folium.GeoJson(data=geo_j, style_function=lambda x: {"fill":False,"color": "red"})#orange"})
geo_j.add_to(m)

for edge in G0.edges():
    c1 = G.nodes[edge[0]]['pos']
    c2 = G.nodes[edge[1]]['pos']
    folium.PolyLine([[c1[1],c1[0]],[c2[1],c2[0]]], weight=1).add_to(m)

for node in G0.nodes():
    if node in ridge_line_dict['sep_group']:

        folium.CircleMarker(location=[G.nodes[node]['pos'][1], G.nodes[node]['pos'][0]],
                            radius=4,
                            color = 'black', tooltip= str(node)+ ' '+ str([G.nodes[node]['pos'][1], G.nodes[node]['pos'][0]])).add_to(m)  #weight=5, color = 'green').add_to(m)

    
        folium.CircleMarker(location=[G.nodes[node]['pos'][1], G.nodes[node]['pos'][0]],
                            radius=0.1,
                            color = 'darkorange', tooltip= str(node)+ ' '+ str([G.nodes[node]['pos'][1], G.nodes[node]['pos'][0]])).add_to(m)  #weight=5, color = 'green').add_to(m)
    
    elif node in ridge_line_dict['group_1']:
        folium.CircleMarker(location=[G.nodes[node]['pos'][1], G.nodes[node]['pos'][0]],
                            radius=0.1,
                            color = 'aqua', tooltip= str(node) + ' '+ str([G.nodes[node]['pos'][1], G.nodes[node]['pos'][0]])).add_to(m)  #weight=5, color = 'green').add_to(m)

    elif node in ridge_line_dict['group_2']:
        folium.CircleMarker(location=[G.nodes[node]['pos'][1], G.nodes[node]['pos'][0]],
                            radius=0.1,
                            color = 'blue', tooltip= str(node) + ' '+ str([G.nodes[node]['pos'][1], G.nodes[node]['pos'][0]])).add_to(m)  #weight=5, color = 'green').add_to(m)


print('Ridge Line Separation')
m

In [ ]:
print('Ridge Line Sparation\n')

ridge_line_group1 =  list(set(ridge_line_dict['group_1']) & set(largest_connected_component))
ridge_line_group2 =  list(set(ridge_line_dict['group_2']) & set(largest_connected_component))
ridge_line_separator_group =  list(set(ridge_line_dict['sep_group']) & set(largest_connected_component))


print('group 1 size: {:d}, \ngroup 2 size: {:d}, \nseparator group size: {:d}\n'.format(len(ridge_line_group1),
                                                                                        len(ridge_line_group2),
                                                                                        len(ridge_line_separator_group)))


print('group 1 %: {:0.2f}, \ngroup 2 %: {:0.2f}, \nseparator group %: {:0.2f}\n'.format(100*(len(ridge_line_group2)/total),
                                                                   100*(len(ridge_line_group1)/total),
                                                                   100*(len(ridge_line_separator_group)/total)))

print('% diff in group size: {:0.2f}, \nnumber difference: {:d}, \nN spearators: {:d}'.format(100*(len(ridge_line_group2) - len(ridge_line_group1))/total, 
                                                                                                     len(ridge_line_group2) - len(ridge_line_group1), 
                                                                                                     len(ridge_line_separator_group)))


In [ ]:
ridge_line_dict['sep_group']

In [ ]:
def haversine(lon1, lat1, lon2, lat2):
    """
    Calculate the great circle distance in kilometers between two points 
    on the earth (specified in decimal degrees)
    """
    # convert decimal degrees to radians 
    lon1, lat1, lon2, lat2 = map(radians, [lon1, lat1, lon2, lat2])

    # haversine formula 
    dlon = lon2 - lon1 
    dlat = lat2 - lat1 
    a = sin(dlat/2)**2 + cos(lat1) * cos(lat2) * sin(dlon/2)**2
    c = 2 * asin(sqrt(a)) 
    r = 3956 #6371 # Radius of earth in kilometers. Use 3956 for miles. Determines return value units.
    return c * r

In [ ]:
active_zone_bounds =  ((-122.00, -121.75),(37.00, 37.25))

print('distance between: [{}, {}] and [{}, {}]'.format(active_zone_bounds[0][0], active_zone_bounds[1][0], 
                                                       active_zone_bounds[0][0], active_zone_bounds[1][1]))

print('{} km'.format(haversine(active_zone_bounds[0][0], active_zone_bounds[1][0], active_zone_bounds[0][0], active_zone_bounds[1][1])))

In [ ]:
print('distance between: [{}, {}] and [{}, {}]'.format(active_zone_bounds[0][0], active_zone_bounds[1][0], 
                                                       active_zone_bounds[0][1], active_zone_bounds[1][0]))

print('{} km'.format(haversine(active_zone_bounds[0][0], active_zone_bounds[1][0], active_zone_bounds[0][1], active_zone_bounds[1][0])))

In [ ]:
## Acres per 'point'

247.105 * (17.261306302223723*13.785488202054845) / (50*62) ## acres ( 247.105 acres = 1 km^2)

### Separate the Target While Maintaining the Rest of the Connectivitiy 
    - geographically disributed XXXX (power or population or ...)
    - removing either assets in a circle or in a defined type of area (census block)
    - want to disrupt functionality to get from A to B while maintaining ability to get from C to D. eg:
        - degrade ability for troops to port while maintaining ability of civilian population to evacuate
        - check worst case scenario for US, want to protect ourselves again this type of attack